# JM2D toy domain (arXiv 2509.08775)

Two sections, both using the public API (`jm2d_toy.samplers`, `jm2d_toy.tasks`):

1. **`donut_joint`** (paper Fig. 2): joint sampling of data $x=(p_1,p_2)$ and parameter $k$ on the donut. Sequential vs Gibbs vs JM2D.
2. **`donut_constrained`** (paper Fig. 9): constrained generation on the donut, scenario `great_wall_rectangle`. Projection vs JM2D.

Every `sample(problem, n, seed)` call is reproducible: it seeds python/numpy/torch, builds the sampler and runs the reverse diffusion. Hyper-parameters come from `configs/*.json`. Run from the repository root (or `notebooks/`).

In [ ]:
import os, sys
ROOT = os.getcwd()  # repository root: the first parent directory that contains jm2d_toy/
while not os.path.isdir(os.path.join(ROOT, 'jm2d_toy')) and os.path.dirname(ROOT) != ROOT:
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)

import jm2d_toy                                    # sets CUBLAS_WORKSPACE_CONFIG before CUDA starts
import matplotlib.pyplot as plt
from jm2d_toy import samplers, tasks
from jm2d_toy.config import load_config
from jm2d_toy.seeding import enable_determinism
from jm2d_toy.plotting import plot_joint_samples, plot_constrained_samples, plot_diffusion_trajectories

enable_determinism()
SEED = 0

## 1. `donut_joint`: joint sampling (Sequential / Gibbs / JM2D)

`DonutJointProblem`: maximise the perimeter of the triangle $(p_1, p_2, k)$ subject to the segments $k\!-\!p_1$ and $k\!-\!p_2$ avoiding four circular obstacles. A sample is *compatible* when the constraint holds. Paper (single draws, N = 16): 2/16, 7/16, 15/16.

In [ ]:
cfg_j = load_config('donut_joint')
problem_j = tasks.make('donut_joint')
res_j = {}
for name in ['sequential', 'gibbs', 'jm2d']:
    sampler = samplers.from_config(cfg_j, name)
    res_j[name] = sampler.sample(problem_j, n=cfg_j['n'], seed=SEED)
    print(f"{name:<11} compatible {int(res_j[name].valid.sum())}/{cfg_j['n']}  ood {int(res_j[name].ood.sum())}/{cfg_j['n']}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5.2))
for ax, (name, r) in zip(axes, res_j.items()):
    plot_joint_samples(ax, problem_j, r.x, r.k, r.valid, marker_scale=0.2,
                    title=f"{name}: {int(r.valid.sum())}/{r.n} compatible")
plt.tight_layout(); plt.show()

## 2. `donut_constrained`: constrained generation (Projection vs JM2D)

`DonutConstrainedProblem`, scenario `great_wall_rectangle`: samples must avoid the wall (red). Projection makes every sample feasible but pushes many of them off the data support (OOD, outside the dashed annulus); JM2D keeps samples in distribution. The projection sampler solves a scipy problem per sample and step (about 2 minutes for N = 64).

In [ ]:
cfg_c = load_config('donut_constrained')
problem_c = tasks.make('donut_constrained')
res_c = {}
for name in ['projection', 'jm2d']:
    sampler = samplers.from_config(cfg_c, name)
    res_c[name] = sampler.sample(problem_c, n=cfg_c['n'], seed=SEED)
    print(f"{name:<11} valid {int(res_c[name].valid.sum())}/{cfg_c['n']}  ood {int(res_c[name].ood.sum())}/{cfg_c['n']}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
for ax, (name, r) in zip(axes, res_c.items()):
    plot_constrained_samples(ax, problem_c, r.x, r.valid, legend=(name == 'jm2d'),
                    title=f"{name}: {int(r.valid.sum())}/{r.n} valid, {int(r.ood.sum())}/{r.n} OOD")
plt.tight_layout(); plt.show()

### Paper Fig. 9(c): one projection trajectory

`return_history=True` records the per-step samples. It evaluates extra network calls that draw random numbers, so the samples differ from a run without history for the same seed.

In [ ]:
traj = samplers.from_config(cfg_c, 'projection').sample(problem_c, n=1, seed=SEED, return_history=True)
fig, ax = plt.subplots(figsize=(7, 6))
plot_diffusion_trajectories(ax, problem_c, traj.history['noisy_samples'])
ax.set_title(f"projection trajectory, final sample OOD: {bool(traj.ood[0])}")
plt.show()

## 3. Stored paper runs and the command line

The runs behind the paper panels are shipped in `results/paper/`; `jm2d_toy.compare.compare` aggregates them (CPU only) and checks them against `paper/targets.json`. The same from the shell:

```bash
python scripts/run.py --task donut_joint --planner sequential gibbs jm2d --seeds 0-4 --out results/runs
python scripts/compare.py --results results/runs --task donut_joint --check-paper --out results/runs/figures/donut_joint.png
python scripts/compare.py --paper --task donut_constrained --check-paper      # stored paper runs
```

In [ ]:
from jm2d_toy.compare import compare
for task in ['donut_joint', 'donut_constrained']:
    _, text = compare(os.path.join(ROOT, 'results', 'paper'), task, check_paper=True)
    print(text, end='\n\n')